<a href="https://colab.research.google.com/github/beatrisAS/Desafio1_API_Streaming_Filmes_Python/blob/main/api/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafio 01 — Conexão Python com Google Sheets API

## Tema 3 · Streaming de filmes · Opção A: prática da sessão

Este notebook segue o guia: autentica uma service account criada no Google Cloud, conecta à Google Sheets API e escreve o cabeçalho e 10 linhas de teste na aba `filmes`.

**Nome da planilha:** `Desafio_01_Streaming_Filmes`  
**Aba/tabela:** `filmes`


## Preparação necessária

No Google Cloud Console:
1. Crie/seleciona um projeto.
2. Ative **Google Sheets API** e **Google Drive API**.
3. Crie uma service account, gere uma chave JSON e guarde o arquivo com segurança.

No Google Sheets: crie a planilha `Desafio_01_Streaming_Filmes`, copie o ID do endereço (entre `/d/` e `/edit`) e compartilhe a planilha com o `client_email` da service account com acesso de **Editor**.

No Colab: abra **Secrets** (ícone de chave), crie o secret `GCP_SA_JSON`, cole nele o conteúdo completo da chave JSON e habilite o acesso pelo notebook. **A chave JSON funciona como senha: não compartilhe nem publique.**


## Célula 1 — instalar bibliotecas


In [14]:
!pip install -q gspread faker


## Célula 2 — importar e autenticar a service account
Na primeira execução, permita o acesso ao secret `GCP_SA_JSON`. Se o Colab pedir, execute a célula novamente.

In [15]:
import json
import random
import gspread
from google.colab import userdata
from faker import Faker

info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
print("Conectado como:", info["client_email"])


Conectado como: desafio1-python@desafio-01-streaming-filmes.iam.gserviceaccount.com


## Célula 3 — abrir a planilha
Substitua o texto de `SHEET_ID` pelo ID da planilha. A primeira aba será renomeada para `filmes`.

In [16]:
SHEET_ID = "1Q6d6Rv2-WS7rz9mwEYbp3MohhXaXzDGy7rIsES-yR-U"
NOME_TABELA = "filmes"

sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
ws.update_title(NOME_TABELA)
print(sh.title, "|", ws.title, "|", sh.url)


Desafio_01_Streaming_Filmes | filmes | https://docs.google.com/spreadsheets/d/1Q6d6Rv2-WS7rz9mwEYbp3MohhXaXzDGy7rIsES-yR-U


## Célula 4 — escrever o cabeçalho do esquema
Use os nomes de coluna exatos da ficha do tema.

In [17]:
COLUNAS = [
    "filme_id",
    "titulo",
    "genero",
    "ano_lancamento",
    "duracao_min",
    "classificacao_indicativa",
    "diretor_ficticio",
]

ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)


Cabeçalho confere: True


## Célula 5 — gerar e escrever 10 linhas de teste
Os valores respeitam o esquema da tabela dimensão `filmes`.

In [18]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

GENEROS = [
    "ação", "comédia", "drama", "terror", "documentário",
    "animação", "ficção científica", "romance"
]
CLASSIFICACOES = ["L", "10", "12", "14", "16", "18"]
ADJETIVOS = ["Sombra", "Distante", "Dourado", "Eterno", "Silencioso", "Último", "Profundo", "Invisível", "Escarlate", "Selvagem"]
SUBSTANTIVOS = ["Vale", "Destino", "Deserto", "Coração", "Horizonte", "Oceano", "Reino", "Caminho", "Cidade", "Espelho"]

def gerar_linha(i):
    # Cria um registro de filme fictício seguindo as sete colunas do esquema.
    return {
        "filme_id": i,
        "titulo": f"{random.choice(ADJETIVOS)} {random.choice(SUBSTANTIVOS)} {i:02d}",
        "genero": random.choice(GENEROS),
        "ano_lancamento": random.randint(1985, 2024),
        "duracao_min": random.randint(75, 184),
        "classificacao_indicativa": random.choice(CLASSIFICACOES),
        "diretor_ficticio": fake.name(),
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; coluna incorreta gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")


10 linhas enviadas


## Célula 6 — ler os dados de volta e conferir
O esperado é 11 linhas na aba: uma linha de cabeçalho e 10 linhas de dados.

In [19]:
valores_lidos = ws.get_all_values()
print("Linhas (cabeçalho + dados):", len(valores_lidos))
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print("Amostra:")
for linha in valores_lidos[:4]:
    print(linha)

assert len(valores_lidos) == 11, "Esperava cabeçalho + 10 linhas de teste."
assert valores_lidos[0] == COLUNAS, "O cabeçalho não corresponde ao esquema."


Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
Amostra:
['filme_id', 'titulo', 'genero', 'ano_lancamento', 'duracao_min', 'classificacao_indicativa', 'diretor_ficticio']
['1', 'Distante Vale 01', 'documentário', '2000', '103', '10', 'Brenda Alves']
['2', 'Distante Cidade 02', 'comédia', '2022', '129', 'L', 'Sra. Isabelly Câmara']
['3', 'Sombra Destino 03', 'terror', '1999', '139', '16', 'Cauã Rocha']


## Conferência final

Abra a planilha no navegador e verifique o cabeçalho na primeira linha, os 10 registros abaixo e os nomes exatos das colunas. Com isso, a prática da sessão está concluída. **Pare aqui:** a carga completa é a etapa de trabalho autônomo descrita na Parte 3 do guia.
